# PIRLS 2021 Digital Reading Process & Interaction Logs (Study B)
## Descriptive Latency Decomposition, Revisit Behavior, and Time-Accuracy Dynamics in Grade 4 Reading (`04_pirls_2021_process_data`)

**Author**: Computational Sketchbook Observatory  
**Date**: October 2026 (Calibrated Edition)  
**Primary Dataset**: IEA PIRLS 2021 International Database (`ASPDEUR5.sav`, `ASADEUR5.sav`, `P21_ItemInformation.xlsx`)  
**Study Scope**: High-income OECD digital reading benchmark sample ($N = 4,486$ clean fourth-grade examinees across 273 digital reading items, yielding 91,376 matched student-item observations).

---

### Executive Research Purpose & Methodological Scope
In Study A, our within-school item-controlled analysis of TIMSS 2019 demonstrated that fourth graders suffer a robust **$-2.73$ percentage-point penalty** on constructed-response mathematics questions under digital administration ($-2.58$ pp among officially invariant items).

However, test-score outcomes alone cannot observe the real-time behavioral process of test-taking. Study B utilizes digital process logs from PIRLS 2021 to examine how young elementary students allocate time, navigate on-screen passages, and formulate responses during computerized assessments.

#### Epistemic Demarcation & Calibrated Interpretation
1. **Descriptive Behavior vs. Causal Transcription Tax**:
   Because paper response latencies are unobserved for these reading items, the observed $+78.5\text{s}$ duration difference between constructed-response and multiple-choice items cannot be identified as a causal "digital transcription tax." A constructed response takes longer than selecting a radio button for multiple cognitive reasons: formulating, organizing, writing, and revising text. Study B provides descriptive evidence of digital test-taking behavior that generates testable hypotheses for mathematics.
2. **Institutional Context**:
   NCES sampled U.S. students for both digital and paper-bridge administration in PIRLS 2021, but published its official results from the paper-bridge sample due to psychometric linking concerns. Germany serves as the primary high-income OECD digital benchmark administering the complete digital PIRLS assessment with verified event timestamps and reading proficiency scales.


## 1. Study Admission Gate & Estimand Specifications

### The Six-Point Study Admission Gate
1. **Retrieval & Usability**: Full access to the public-use PIRLS 2021 International Database (IEA Hamburg), including process timing files (`ASP`) and scored achievement microdata (`ASA`).
2. **Analytical Population**: Exactly $N = 4,486$ clean students meeting official participation criteria (`FLAGPT1 == 0` and `FLAGPT2 == 0`) across 273 digital reading items (124 Multiple Choice, 149 Constructed Response).
3. **Explicit Estimand Labeling**:
   - **Table 15**: Item-level macro-averages (unweighted across 273 items).
   - **Table 16**: Student-level population-weighted averages (using sampling weights `TOTWGT`, $N = 4,486$).
   - **Table 17**: Student-item matched panel regressions with item fixed effects ($N = 91,376$ observations).
4. **Directional Neutrality**: We test competing behavioral hypotheses without presupposing an interface penalty:
   - Does navigation frequency explain format differences?
   - Does extra time on task predict higher accuracy (productive engagement) or lower accuracy (hesitation/confusion)?
5. **Audited Data Integrity**: Pinned dependencies, verified two-digit scoring keys, and recovered timing fields.


In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "artifacts" / "tables"

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 1000)

df_items = pd.read_parquet(PROCESSED_DIR / "pirls_2021_item_process_metrics.parquet")
clean_stu = pd.read_parquet(PROCESSED_DIR / "pirls_2021_student_process_sample.parquet")
df_panel = pd.read_parquet(PROCESSED_DIR / "pirls_2021_student_item_panel.parquet")

print(f"[OK] Loaded {len(df_items)} digital items ({df_items['item_type'].value_counts().to_dict()})")
print(f"[OK] Loaded {len(clean_stu)} clean examinees (Mean PV1: {clean_stu['ASRREA01'].mean():.1f} pts)")
print(f"[OK] Loaded {len(df_panel):,} matched student-item observations across {df_panel['item_id'].nunique()} items")


[OK] Loaded 273 digital items ({'CR': 149, 'MC': 124})
[OK] Loaded 4486 clean examinees (Mean PV1: 527.1 pts)
[OK] Loaded 91,376 matched student-item observations across 273 items


## 2. Item-Level Response Latency Decomposition (Table 15)

We calculate unweighted macro-averages across 273 digital reading items (124 Multiple Choice, 149 Constructed Response) across all four reading comprehension processes:
1. *Focus on and Retrieve Explicitly Stated Information*
2. *Make Straightforward Inferences*
3. *Interpret and Integrate Ideas and Information*
4. *Evaluate and Critique Content and Textual Elements*


In [2]:
df_t15 = pd.read_csv(TABLES_DIR / "table15_pirls_2021_process_metrics_by_format.csv")
display(df_t15)

# Highlight Key Ratios
mc_ov = df_t15[(df_t15["grouping"]=="Overall Format") & (df_t15["category"].str.contains("Multiple Choice"))].iloc[0]
cr_ov = df_t15[(df_t15["grouping"]=="Overall Format") & (df_t15["category"].str.contains("Constructed Response"))].iloc[0]

print("\n--- Overall Item Latency Contrast (Table 15 Macro-Averages) ---")
print(f"Multiple Choice Mean Total Time:         {mc_ov['mean_total_time_s']:.1f} seconds")
print(f"Constructed Response Mean Total Time:     {cr_ov['mean_total_time_s']:.1f} seconds")
print(f"Observed Format Duration Wedge:           +{cr_ov['mean_total_time_s'] - mc_ov['mean_total_time_s']:.1f} seconds per item")
print(f"Latency Multiplier:                       {cr_ov['entry_latency_ratio']:.2f}x")
print(f"Revisit Rate Gap:                         {cr_ov['revisit_pct']:.1f}% CR vs {mc_ov['revisit_pct']:.1f}% MC (Delta = +{cr_ov['revisit_pct'] - mc_ov['revisit_pct']:.1f} pp)")
print(f"Additional Time After First Visit:        {cr_ov['mean_additional_time_after_first_visit_s']:.1f}s CR vs {mc_ov['mean_additional_time_after_first_visit_s']:.1f}s MC (3.3x)")
print(f"Conditional Revisit Duration (when F>1):  {cr_ov['mean_conditional_revisit_duration_s']:.1f}s CR vs {mc_ov['mean_conditional_revisit_duration_s']:.1f}s MC (3.1x)")


,grouping,category,n_items,mean_total_time_s,median_total_time_s,mean_first_visit_s,revisit_pct,mean_revisit_duration_s,mean_additional_time_after_first_visit_s,mean_conditional_revisit_duration_s,entry_latency_ratio,estimand_note
0,Overall Format,Multiple Choice (Selected Response),124,46.90,30.0,37.71,49.78,9.19,9.19,18.53,1.00,Macro-average across items (unweighted) on cle...
1,Overall Format,Constructed Response (Open Ended),149,123.12,89.5,92.74,52.19,30.38,30.38,56.85,2.63,Macro-average across items (unweighted) on cle...
2,Comprehension Process,Evaluate and Critique Content and Textual Elem...,20,49.83,35.5,40.65,48.89,9.17,9.17,19.03,1.00,Macro-average across 20 items in Evaluate and ...
3,Comprehension Process,Evaluate and Critique Content and Textual Elem...,19,119.32,87.0,85.29,54.61,34.02,34.02,60.85,2.39,Macro-average across 19 items in Evaluate and ...
4,Comprehension Process,Focus on and Retrieve Explicitly Stated Inform...,40,42.79,24.5,34.37,48.68,8.42,8.42,17.45,1.00,Macro-average across 40 items in Focus on and ...
5,Comprehension Process,Focus on and Retrieve Explicitly Stated Inform...,48,104.75,73.5,82.53,49.13,22.22,22.22,44.92,2.45,Macro-average across 48 items in Focus on and ...
6,Comprehension Process,Interpret and Integrate Ideas and Information ...,15,45.59,29.0,36.64,51.11,8.94,8.94,17.52,1.00,Macro-average across 15 items in Interpret and...
7,Comprehension Process,Interpret and Integrate Ideas and Information ...,53,134.80,106.0,99.70,54.79,35.10,35.10,62.29,2.96,Macro-average across 53 items in Interpret and...
8,Comprehension Process,Make Straightforward Inferences (MC),49,49.46,32.0,39.56,50.63,9.90,9.90,19.51,1.00,Macro-average across 49 items in Make Straight...
9,Comprehension Process,Make Straightforward Inferences (CR),29,134.69,111.5,101.81,50.93,32.88,32.88,64.00,2.72,Macro-average across 29 items in Make Straight...



--- Overall Item Latency Contrast (Table 15 Macro-Averages) ---
Multiple Choice Mean Total Time:         46.9 seconds
Constructed Response Mean Total Time:     123.1 seconds
Observed Format Duration Wedge:           +76.2 seconds per item
Latency Multiplier:                       2.63x
Revisit Rate Gap:                         52.2% CR vs 49.8% MC (Delta = +2.4 pp)
Additional Time After First Visit:        30.4s CR vs 9.2s MC (3.3x)
Conditional Revisit Duration (when F>1):  56.9s CR vs 18.5s MC (3.1x)


## 3. Empirical Evaluation of Behavioral Hypotheses (Table 16)

### Hypothesis 1: Navigation Friction (Paging & Scrolling)
- **Prediction**: If navigation friction disproportionately harms constructed response performance, examinees will exhibit substantially higher revisit rates on CR items due to interface confusion.
- **Empirical Reality**: Revisit rates are virtually identical across formats: **52.2%** on CR vs. **49.8%** on MC ($\Delta = +2.4\text{ pp}$).
- **Behavioral Nuance**: Revisit duration is $3.1\times$ to $3.3\times$ longer on CR ($30.4\text{s}$ vs $9.2\text{s}$ unconditional; $56.8\text{s}$ vs $18.5\text{s}$ conditional). This indicates that revisits to CR items involve substantive answer drafting and editing, not screen disorientation.

### Hypothesis 2: Digital Response Duration (Observed Format Wedge)
- **Empirical Reality**: Students spend a population-weighted average of **127.0 seconds** per CR item versus **48.6 seconds** per MC item—a **$+78.4\text{s}$ per-item duration wedge** ($2.61\times$).
- **Calibrated Interpretation**: In a 40-minute test block (12–14 items), CR questions absorb nearly triple the temporal budget of MC questions. However, because paper completion times are unobserved, this reflects the general cognitive and motor demands of constructing text rather than an isolated keyboarding penalty.

### Hypothesis 3: General Academic Ability (Comprehension Moderating Duration)
- **Empirical Reality**:
  - $r(\text{MC Time}, \text{Reading Score}) = -0.094$: Proficient readers answer multiple-choice questions faster.
  - $r(\text{CR Time}, \text{Reading Score}) = +0.114$: Proficient readers spend *longer* on constructed-response questions.
  - $r(\text{Wedge}, \text{Reading Score}) = +0.173$: Higher academic ability widens the response duration wedge, as stronger students write longer, multi-sentence answers.


In [3]:
df_t16 = pd.read_csv(TABLES_DIR / "table16_pirls_2021_mechanism_decomposition.csv")
for _, r in df_t16.iterrows():
    print(f"\n================================================================================")
    print(f"{r['mechanism_hypothesis']}")
    print(f"================================================================================")
    print(f"Metric:        {r['empirical_metric']}")
    print(f"Observed:      {r['observed_value']}")
    print(f"Contrast:      {r['p_value_or_wedge']}")
    print(f"Evaluation:    {r['substantive_evaluation']}")
    print(f"Estimand:      {r['estimand_note']}")



Hypothesis 1: Navigation Friction (Paging & Revisit Burden)
Metric:        Revisit Probability (%) & Duration Beyond First Visit (s)
Observed:      Revisit Rate: 52.2% CR vs 49.8% MC (+2.4%); Additional Time: 30.4s CR vs 9.2s MC (3.3x); Conditional Revisit: 56.8s vs 18.5s (3.1x)
Contrast:      Revisit rate gap = +2.4 pp (minimal format difference); Additional time wedge = +21.2s
Evaluation:    Supported for duration, format-neutral for frequency: Fourth graders revisit MC and CR items at virtually identical rates (~50-52%), but spend 3x longer on revisits to CR items, reflecting answer drafting/revising rather than navigational disorientation.
Estimand:      Item-level unweighted macro-average across 273 items on clean sample (N=4,486)

Hypothesis 2: Digital Response Duration (Observed CR vs MC Wedge)
Metric:        Mean Total Response Time Wedge (CR - MC seconds)
Observed:      CR Weighted Mean: 127.2s vs MC Weighted Mean: 48.7s per item
Contrast:      Wedge = +78.5 s (2.61x response

## 4. Within-Item Time-Accuracy Gradient (Table 17)

To test whether extra time spent on an item represents **productive task engagement** (higher score) or **unproductive hesitation/struggle** (lower score), we estimate within-item fixed effects regressions on the 91,376 matched student-item observations:

$$Y_{ij} - \bar{Y}_j = \beta \cdot (\ln T_{ij} - \overline{\ln T}_j) + \epsilon_{ij}$$

Where $Y_{ij}$ is the percent score earned by student $i$ on item $j$, and $T_{ij}$ is the total time spent in seconds.


In [4]:
df_t17 = pd.read_csv(TABLES_DIR / "table17_pirls_2021_time_accuracy_decomposition.csv")
display(df_t17)

print("\n--- Key Finding from Within-Item Matched Analysis ---")
mc_row = df_t17[df_t17["item_format"].str.startswith("Multiple Choice")].iloc[0]
cr_row = df_t17[df_t17["item_format"].str.startswith("Constructed Response")].iloc[0]

print(f"Multiple Choice Within-Item Beta:       {mc_row['within_item_beta_log_time']:+.4f} (Corr: {mc_row['within_item_corr_log_time']:+.4f})")
print(f"Constructed Response Within-Item Beta:  {cr_row['within_item_beta_log_time']:+.4f} (Corr: {cr_row['within_item_corr_log_time']:+.4f})")
print(f"Format Slope Wedge:                     {cr_row['within_item_beta_log_time'] - mc_row['within_item_beta_log_time']:+.4f}")


,item_format,n_observations,n_items,mean_time_s,mean_score,within_item_corr_log_time,within_item_beta_log_time,within_item_beta_raw_time_per_min,substantive_interpretation
0,Multiple Choice (Selected Response),43658,124,46.9,71.3,-0.0595,-0.0327,-0.0191,Negative gradient: Extra time spent reflects d...
1,Constructed Response (Open Ended),47718,149,130.5,61.7,0.0660,0.0408,0.0112,Positive gradient: Extra time spent reflects p...



--- Key Finding from Within-Item Matched Analysis ---
Multiple Choice Within-Item Beta:       -0.0327 (Corr: -0.0595)
Constructed Response Within-Item Beta:  +0.0408 (Corr: +0.0660)
Format Slope Wedge:                     +0.0735


## 5. Student Pacing & Reading Ability Distribution

We examine how student pacing and accuracy trade off across quartiles of reading achievement.


In [5]:
clean_stu["reading_quartile"] = pd.qcut(clean_stu["ASRREA01"], 4, labels=["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"])
pacing_by_q = clean_stu.groupby("reading_quartile").agg(
    mean_pv1=("ASRREA01", "mean"),
    mc_acc_pct=("mc_score_pct", "mean"),
    cr_acc_pct=("cr_score_pct", "mean"),
    mean_mc_s=("mean_mc_time_s", "mean"),
    mean_cr_s=("mean_cr_time_s", "mean"),
    mean_wedge_s=("cr_mc_time_diff_s", "mean")
)
print("--- Pacing and Accuracy Metrics by Reading Proficiency Quartile ---")
display(pacing_by_q.round(2))


--- Pacing and Accuracy Metrics by Reading Proficiency Quartile ---


C:\Users\admir\AppData\Local\Temp\ipykernel_15276\2610390358.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pacing_by_q = clean_stu.groupby("reading_quartile").agg(


,mean_pv1,mc_acc_pct,cr_acc_pct,mean_mc_s,mean_cr_s,mean_wedge_s
reading_quartile,,,,,,
Q1 (Lowest),425.89,41.78,34.46,49.72,117.40,67.68
Q2,506.61,66.16,55.97,51.93,125.01,73.08
Q3,556.56,80.68,69.01,49.92,131.90,81.97
Q4 (Highest),619.47,90.43,81.01,42.90,133.22,90.31


## 6. Synthesis: Generating Hypotheses for U.S. Mathematics

Study B provides crucial behavioral context for interpreting the digital constructed-response penalty identified in TIMSS 2019 Study A:

1. **Constructed Responses Demand Heavy Time Investment**:
   Digital open-ended questions require **$2.6\times$ more time than multiple choice** (+78.5s per item). On digital tests, constructed response is an intrinsically high-latency activity.
2. **Extra Time on Constructed Response is Productive**:
   Unlike multiple-choice items—where longer duration signals confusion or guessing ($\beta = -0.033$)—longer duration on constructed-response items associates with **higher scores** ($\beta = +0.041$). Students earn credit by drafting, expanding, and refining their answers.
3. **The Keyboarding Hypothesis**:
   If productive performance on constructed responses requires extensive drafting, students who lack keyboarding fluency or input automaticity face a binding constraint. Under timed test conditions, typing friction may prevent students from fully articulating what they understand.
4. **Next Steps in the Empirical Agenda**:
   - **Longitudinal Trend Study**: Did U.S. fourth-grade performance on common digital constructed-response mathematics items decline disproportionately between TIMSS 2019 and 2023?
   - **Randomized Assessment Trial**: Does an experimental intervention (Paper vs. Screen vs. Digital Entry vs. Authentic Practice) eliminate the response-entry wedge?
